# 02 — Silver: Yellow Taxi

Reads Yellow Taxi records from Bronze (`tlc_bronze.yellow_raw`), applies data
quality rules via **flag-based annotation** (no silent drops), enriches with
zone metadata, builds the normalized Silver schema, and writes:
- **Valid records** → `tlc_silver.trips_yellow` (with `quality_flags` preserved)
- **Invalid records** → `tlc_audit.quarantine` (with `_rejection_reason`)

Silver `_meta` carries `bronze_run_id` + `source_file` for full lineage tracing.

## Imports & Configuration

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from src.transformations.tlc_rules import (
    YELLOW_GREEN_RULES, apply_quality_flags, route_quarantine, print_rejection_summary
)
from src.transformations.enrichment import load_zone_lookup, enrich_trip_locations
from src.transformations.schema import build_yellow_silver
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
from core.storage.mongo_client import get_audit_db
import pyspark.sql.functions as F
import datetime

YEARS_TO_PROCESS = [2024, 2025]
print("Imports OK.")

## Start Spark & Audit Control

In [ ]:
spark = get_spark(app_name="TLC_Silver_Yellow")

control = ControlManager("silver_yellow")
execution = control.start({"years": YEARS_TO_PROCESS, "vehicle_type": "yellow"})
run_id = execution.execution_id
print(f"Execution ID (Silver run_id): {run_id}")

## Read from Bronze

In [ ]:
df_bronze = read_mongo(spark, settings.MONGO_DB_BRONZE, "yellow_raw")

# Filter to requested years (safe: static batch DataFrame)
if "_meta" in df_bronze.columns:
    df_bronze = df_bronze.filter(F.year("tpep_pickup_datetime").isin(YEARS_TO_PROCESS))

records_in = df_bronze.count()
print(f"Records read from Bronze: {records_in:,}")

## Apply Data Quality Flags

> **No records are dropped here.**  
> Each record receives a `quality_flags` struct (one boolean per rule) and an
> `is_valid` column. Invalid records are *routed* to quarantine below.

In [ ]:
# Step 1 — annotate every record with quality flags (no filtering)
df_flagged = apply_quality_flags(df_bronze, YELLOW_GREEN_RULES)

# Step 2 — split into valid / rejected partitions
valid_df, rejected_df = route_quarantine(df_flagged)

records_valid    = valid_df.count()
records_rejected = rejected_df.count()

print(f"Valid records   : {records_valid:,}")
print(f"Rejected records: {records_rejected:,}")
print(f"Quarantine rate : {records_rejected / records_in * 100:.2f}%" if records_in else "N/A")

print_rejection_summary(rejected_df)

control.log_quality_check(
    check_name="yellow_quality_flags",
    dataset=f"yellow_raw_years_{YEARS_TO_PROCESS}",
    records_checked=records_in,
    records_failed=records_rejected,
)

## Route Rejected Records to Quarantine

Invalid records are stored in `tlc_audit.quarantine` with:
- `_rejection_reason` — human-readable failing rule
- `quality_flags` — full per-rule boolean breakdown
- `bronze_run_id` (via `_meta.run_id`) — Bronze lineage
- `pipeline` and `silver_run_id` — Silver lineage

In [ ]:
if records_rejected > 0:
    audit_db = get_audit_db()
    quarantine_col = audit_db["quarantine"]

    quarantine_docs = [
        {
            "quarantined_at": datetime.datetime.utcnow(),
            "silver_run_id":  run_id,
            "bronze_run_id":  row.asDict(recursive=True).get("_meta", {}).get("run_id"),
            "source_file":    row.asDict(recursive=True).get("_meta", {}).get("source_file"),
            "pipeline":       "silver_yellow",
            "reason":         row["_rejection_reason"],
            "quality_flags":  row.asDict(recursive=True).get("quality_flags", {}),
            "raw_record":     {
                k: v for k, v in row.asDict().items()
                if k not in ("_rejection_reason", "quality_flags")
            },
        }
        for row in rejected_df.limit(10000).collect()  # cap at 10k for large batches
    ]
    if quarantine_docs:
        quarantine_col.insert_many(quarantine_docs)
        print(f"Quarantined {len(quarantine_docs):,} records into tlc_audit.quarantine")
else:
    print("No records quarantined.")

## Enrich with Zone Metadata

In [ ]:
zone_df = load_zone_lookup(spark)
valid_df = enrich_trip_locations(valid_df, zone_df)
print("Zone enrichment complete.")

## Build Silver Schema & Write to MongoDB

Silver documents include:
- `quality_flags` — the per-rule audit trail
- `_meta.bronze_run_id` — traces back to the Bronze ingestion run
- `_meta.source_file` — original raw Parquet filename

In [ ]:
silver_df = build_yellow_silver(valid_df, run_id=run_id)
n_written = write_mongo(silver_df, settings.MONGO_DB_SILVER, "trips_yellow", mode="append")
print(f"Rows written to tlc_silver.trips_yellow: {n_written:,}")
print(f"Lineage: Bronze → Silver run_id={run_id}")

## Volumetric Traceability Check

Validates: **Bronze == Silver + Quarantine** (zero data lost in transit)

In [ ]:
print(f"╔══════════════════════════════════════════╗")
print(f"║  Volumetric Traceability Report          ║")
print(f"╠══════════════════════════════════════════╣")
print(f"║  Bronze records in  : {records_in:>16,}  ║")
print(f"║  Silver records out : {n_written:>16,}  ║")
print(f"║  Quarantine records : {records_rejected:>16,}  ║")
print(f"║  Delta (must be 0)  : {records_in - n_written - records_rejected:>16,}  ║")
print(f"╚══════════════════════════════════════════╝")
assert records_in == n_written + records_rejected, "DATA LOSS DETECTED — investigate immediately!"

In [ ]:
control.end(
    ExecutionStatus.SUCCESS,
    {
        "records_read_from_bronze": records_in,
        "records_written_to_silver": n_written,
        "records_quarantined": records_rejected,
        "quarantine_rate_pct": round(records_rejected / records_in * 100, 4) if records_in else 0,
    },
)

## Audit Report

In [ ]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))